# YOLO11 pretrained baseline evaluation
Evaluates pretrained YOLO11n on the frozen custom MIO-TCD **test** split only. It does not train or tune any threshold. Output: Ultralytics artefacts under `outputs/mio_tcd/baseline/` and `baseline_metrics.csv`.

## Config

In [ ]:
MODEL_NAME = 'yolo11n.pt'
IMG_SIZE = 640
DEVICE = None  # None lets Ultralytics choose; use 'cpu', '0', etc.
BATCH_SIZE = 16
WORKERS = 4
SEED = 42
PROJECT_DIR = 'outputs/mio_tcd/baseline'
RUN_NAME = 'yolo11n_pretrained_test'
FORCE_OVERWRITE = False

## Imports and validation

In [ ]:
from pathlib import Path
import random, sys, numpy as np, pandas as pd, torch, ultralytics
from ultralytics import YOLO
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT, metric_tables, prepare_coco_baseline_eval
PROJECT_DATA_YAML = PROJECT_ROOT / 'data/mio_tcd/yolo/mio_tcd.yaml'
MANIFEST_PATH = PROJECT_ROOT / 'data/mio_tcd/splits/split_manifest.csv'
TEST_LIST = PROJECT_ROOT / 'data/mio_tcd/splits/test.txt'
if not PROJECT_DATA_YAML.is_file() or not MANIFEST_PATH.is_file() or not TEST_LIST.is_file(): raise FileNotFoundError('Run 01–03 notebooks first.')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'Ultralytics:', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(), 'device:', DEVICE or ('cuda' if torch.cuda.is_available() else 'cpu'))

## Evaluation

In [ ]:
run_dir = PROJECT_ROOT / PROJECT_DIR
if (run_dir / RUN_NAME).exists() and not FORCE_OVERWRITE: raise FileExistsError('Run directory exists; choose a new RUN_NAME or set FORCE_OVERWRITE=True.')
model = YOLO(MODEL_NAME)
manifest = pd.read_csv(MANIFEST_PATH, dtype={'image_id': str})
DATA_YAML = prepare_coco_baseline_eval(manifest, PROJECT_ROOT / 'data/mio_tcd/yolo', model.names, FORCE_OVERWRITE)
print('COCO-ID baseline YAML:', DATA_YAML)
results = model.val(data=str(DATA_YAML), split='test', imgsz=IMG_SIZE, batch=BATCH_SIZE, workers=WORKERS, device=DEVICE, project=str(run_dir), name=RUN_NAME, exist_ok=FORCE_OVERWRITE, plots=True, verbose=True)
overall, per_class = metric_tables(results, MODEL_NAME)
display(overall); display(per_class)
print('Rare-class metrics:'); display(per_class[per_class['class'].isin(['bicycle', 'motorcycle'])])

## Save summary

In [ ]:
out = PROJECT_ROOT / 'outputs/mio_tcd/baseline_metrics.csv'; out.parent.mkdir(parents=True, exist_ok=True)
if out.exists() and not FORCE_OVERWRITE: raise FileExistsError(f'{out} exists; set FORCE_OVERWRITE=True to replace it.')
overall.to_csv(out, index=False); per_class.to_csv(out.with_name('baseline_per_class_metrics.csv'), index=False)
print('Saved:', out)